# 01 Data Understanding

# Health Insurance Fraud Detection — Data Understanding

## Business objective

The objective of this project is to build a decision-support system that helps fraud investigation teams prioritize health insurance claims for manual review.

For each newly submitted claim, the system will estimate a fraud risk:

\[
P(\text{fraud} \mid \text{information available at scoring time})
\]

The model is not intended to automatically reject claims.

Instead, its output will be used to:

1. rank claims by fraud risk;
2. identify claims that deserve priority investigation;
3. provide interpretable reasons behind elevated risk;
4. support human investigators in allocating limited investigation capacity.

The unit of prediction is **one health insurance claim**.

The target variable is:

- `is_fraud = 1`: fraudulent claim;
- `is_fraud = 0`: legitimate claim.

Because fraud is rare, model performance will not be assessed primarily through accuracy. Evaluation will focus on metrics adapted to class imbalance and operational investigation capacity, including precision, recall, PR-AUC and ranking-based metrics such as Precision@K and Recall@K.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "interim"

CUSTOMERS_PATH = DATA_DIR / "customers.parquet"
PROVIDERS_PATH = DATA_DIR / "providers.parquet"
POLICIES_PATH = DATA_DIR / "policies.parquet"
CLAIMS_PATH = DATA_DIR / "claims.parquet"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

Project root: /workspaces/Fraud-risk-model
Data directory: /workspaces/Fraud-risk-model/data/interim


In [3]:
customers = pd.read_parquet(CUSTOMERS_PATH)
providers = pd.read_parquet(PROVIDERS_PATH)
policies = pd.read_parquet(POLICIES_PATH)
claims = pd.read_parquet(CLAIMS_PATH)

tables = {
    "customers": customers,
    "providers": providers,
    "policies": policies,
    "claims": claims,
}

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [4]:
dataset_overview = pd.DataFrame(
    {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "memory_mb": df.memory_usage(
            deep=True
        ).sum() / 1024**2,
    }
    for name, df in tables.items()
)

dataset_overview

,table,rows,columns,memory_mb
0,customers,20000,6,3.757
1,providers,1200,6,0.282
2,policies,20000,5,3.639
3,claims,99888,55,112.809


## Data model

The dataset is organized around four entities:

- **customers**: insured individuals;
- **policies**: health insurance contracts;
- **providers**: healthcare professionals or organizations;
- **claims**: reimbursement requests submitted to the insurer.

`claims` is the central analytical table.

The main relationships are:

```text
CUSTOMERS
    │
    ├──────────► POLICIES
    │               │
    │               ▼
    └────────────► CLAIMS ◄──────────── PROVIDERS

In [5]:
claims.head()

,claim_id,customer_id,policy_id,provider_id,service_category,service_code,service_units,service_date,claim_submission_date,claim_submission_timestamp,claim_amount,requested_reimbursement,coverage_limit,submission_channel,document_count,has_invoice,has_prescription,customer_age,customer_tenure_months,coverage_level,customer_behavior_segment,policy_tenure_months,recent_policy_change,days_since_policy_change,provider_type,provider_region,provider_tenure_months,provider_behavior_segment,days_service_to_submission,reimbursement_ratio,customer_claims_7d,customer_claims_30d,customer_claims_90d,customer_claims_365d,customer_amount_30d,customer_amount_365d,customer_avg_claim_amount_365d,days_since_customer_previous_claim,days_since_same_provider_claim,customer_provider_claims_30d,same_service_claims_30d,provider_claims_30d,provider_claims_90d,provider_avg_claim_amount_90d,service_typical_amount,claim_to_service_median_ratio,claim_to_customer_avg_ratio,claim_to_provider_avg_ratio,legitimate_anomaly,legitimate_anomaly_type,latent_fraud_score,synthetic_fraud_probability,fraud_difficulty,fraud_mechanism,is_fraud
0,CLM_00075763,CUST_004539,POL_004539,PROV_00107,diagnostic,DIAG_IMAGING,1,2022-12-30,2023-01-01,2023-01-01 00:22:09,141.260,107.150,250.000,web,6.000,True,True,27,33,basic,regular,3,False,NaN,specialist,west,225,normal,2,0.759,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,140.000,1.009,NaN,NaN,False,none,-7.504,0.001,none,none,0
1,CLM_00079340,CUST_009396,POL_009396,PROV_00554,optical,OPT_CONTACT,1,2022-12-15,2023-01-01,2023-01-01 01:19:44,361.960,300.000,300.000,web,6.000,False,False,48,42,basic,regular,35,True,52.000,optician,south,130,normal,17,0.829,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,260.000,1.392,NaN,NaN,False,none,-4.022,0.018,none,none,0
2,CLM_00094443,CUST_009030,POL_009030,PROV_00240,medical_device,DEVICE_OTHER,2,2022-12-12,2023-01-01,2023-01-01 01:34:19,914.860,737.790,"1,000.000",web,4.000,True,False,58,38,standard,regular,5,False,NaN,pharmacy,west,118,normal,20,0.806,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,450.000,2.033,NaN,NaN,False,none,-6.518,0.001,none,none,0
3,CLM_00080995,CUST_015957,POL_015957,PROV_00545,physiotherapy,PHYSIO_STANDARD,5,2022-12-30,2023-01-01,2023-01-01 02:11:21,148.540,82.240,600.000,provider_direct,6.000,True,False,57,104,standard,regular,31,False,NaN,physiotherapist,east,44,normal,2,0.554,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,180.000,0.825,NaN,NaN,False,none,-5.066,0.006,none,none,0
4,CLM_00068752,CUST_017535,POL_017535,PROV_00379,medical_device,DEVICE_HEARING,3,2022-12-23,2023-01-01,2023-01-01 02:24:11,"1,592.740","1,000.000","1,000.000",web,6.000,True,False,70,79,standard,high_usage,49,False,NaN,pharmacy,east,34,high_volume,9,0.628,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,450.000,3.539,NaN,NaN,True,high_amount_legitimate,-6.098,0.002,none,none,0


In [6]:
claims.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99888 entries, 0 to 99887
Data columns (total 55 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   claim_id                            99888 non-null  object        
 1   customer_id                         99888 non-null  object        
 2   policy_id                           99888 non-null  object        
 3   provider_id                         98935 non-null  object        
 4   service_category                    99888 non-null  object        
 5   service_code                        99888 non-null  object        
 6   service_units                       99888 non-null  int64         
 7   service_date                        99888 non-null  datetime64[ns]
 8   claim_submission_date               99888 non-null  datetime64[ns]
 9   claim_submission_timestamp          99888 non-null  datetime64[ns]
 10  claim_amount          

In [7]:
claims.columns.tolist()

['claim_id',
 'customer_id',
 'policy_id',
 'provider_id',
 'service_category',
 'service_code',
 'service_units',
 'service_date',
 'claim_submission_date',
 'claim_submission_timestamp',
 'claim_amount',
 'requested_reimbursement',
 'coverage_limit',
 'submission_channel',
 'document_count',
 'has_invoice',
 'has_prescription',
 'customer_age',
 'customer_tenure_months',
 'coverage_level',
 'customer_behavior_segment',
 'policy_tenure_months',
 'recent_policy_change',
 'days_since_policy_change',
 'provider_type',
 'provider_region',
 'provider_tenure_months',
 'provider_behavior_segment',
 'days_service_to_submission',
 'reimbursement_ratio',
 'customer_claims_7d',
 'customer_claims_30d',
 'customer_claims_90d',
 'customer_claims_365d',
 'customer_amount_30d',
 'customer_amount_365d',
 'customer_avg_claim_amount_365d',
 'days_since_customer_previous_claim',
 'days_since_same_provider_claim',
 'customer_provider_claims_30d',
 'same_service_claims_30d',
 'provider_claims_30d',
 'provi

In [8]:
column_groups = {
    "identifiers": [
        "claim_id",
        "customer_id",
        "policy_id",
        "provider_id",
    ],

    "target": [
        "is_fraud",
    ],

    "temporal": [
        col
        for col in claims.columns
        if "date" in col.lower()
        or "timestamp" in col.lower()
    ],

    "synthetic_metadata": [
        col
        for col in [
            "synthetic_fraud_probability",
            "latent_fraud_score",
            "fraud_mechanism",
            "fraud_difficulty",
        ]
        if col in claims.columns
    ],
}

for group, columns in column_groups.items():
    print(f"\n{group.upper()}")
    print("-" * 50)

    for column in columns:
        print(column)


IDENTIFIERS
--------------------------------------------------
claim_id
customer_id
policy_id
provider_id

TARGET
--------------------------------------------------
is_fraud

TEMPORAL
--------------------------------------------------
service_date
claim_submission_date
claim_submission_timestamp

SYNTHETIC_METADATA
--------------------------------------------------
synthetic_fraud_probability
latent_fraud_score
fraud_mechanism
fraud_difficulty


## Leakage policy

A fraud detection model must only use information that would be available when the claim is scored.

Several columns in this synthetic dataset exist only because the data generation process knows the hidden fraud mechanism.

These variables must never be used as model inputs:

- `is_fraud` — prediction target;
- `synthetic_fraud_probability` — probability used by the synthetic generator;
- `latent_fraud_score` — hidden synthetic fraud signal;
- `fraud_mechanism` — fraud mechanism assigned by the generator;
- `fraud_difficulty` — synthetic difficulty label.

Using any of these variables as predictors would introduce **target leakage** and produce unrealistically optimistic model performance.

Identifiers such as `claim_id` will also not be used directly as predictive features.

Historical features require additional attention: they must be calculated using only information available **strictly before the claim being scored**.

In [9]:
target_distribution = (
    claims["is_fraud"]
    .value_counts(dropna=False)
    .rename_axis("is_fraud")
    .to_frame("count")
)

target_distribution["percentage"] = (
    100
    * target_distribution["count"]
    / len(claims)
)

target_distribution

,count,percentage
is_fraud,,
0,97406,97.515
1,2482,2.485


In [10]:
fraud_rate = claims["is_fraud"].mean()

print(f"Claims: {len(claims):,}")
print(f"Fraud cases: {claims['is_fraud'].sum():,}")
print(f"Fraud prevalence: {fraud_rate:.3%}")

Claims: 99,888
Fraud cases: 2,482
Fraud prevalence: 2.485%


In [11]:
print(
    "First submission:",
    claims["claim_submission_timestamp"].min(),
)

print(
    "Last submission:",
    claims["claim_submission_timestamp"].max(),
)

print(
    "Temporal span:",
    claims["claim_submission_timestamp"].max()
    - claims["claim_submission_timestamp"].min(),
)

First submission: 2023-01-01 00:22:09
Last submission: 2026-06-30 23:58:50
Temporal span: 1276 days 23:36:41


In [12]:
for i, column in enumerate(claims.columns, start=1):
    print(f"{i:02d}. {column}")

01. claim_id
02. customer_id
03. policy_id
04. provider_id
05. service_category
06. service_code
07. service_units
08. service_date
09. claim_submission_date
10. claim_submission_timestamp
11. claim_amount
12. requested_reimbursement
13. coverage_limit
14. submission_channel
15. document_count
16. has_invoice
17. has_prescription
18. customer_age
19. customer_tenure_months
20. coverage_level
21. customer_behavior_segment
22. policy_tenure_months
23. recent_policy_change
24. days_since_policy_change
25. provider_type
26. provider_region
27. provider_tenure_months
28. provider_behavior_segment
29. days_service_to_submission
30. reimbursement_ratio
31. customer_claims_7d
32. customer_claims_30d
33. customer_claims_90d
34. customer_claims_365d
35. customer_amount_30d
36. customer_amount_365d
37. customer_avg_claim_amount_365d
38. days_since_customer_previous_claim
39. days_since_same_provider_claim
40. customer_provider_claims_30d
41. same_service_claims_30d
42. provider_claims_30d
43. pro

|  # | Variable                             | Nature               | Modèle ? | Décision                          |
| -: | ------------------------------------ | -------------------- | :------: | --------------------------------- |
|  1 | `claim_id`                           | identifiant claim    |     ❌    | exclure du modèle                 |
|  2 | `customer_id`                        | identifiant client   |    ⚠️    | exclure comme feature directe     |
|  3 | `policy_id`                          | identifiant contrat  |     ❌    | exclure comme feature directe     |
|  4 | `provider_id`                        | identifiant provider |    ⚠️    | ne pas utiliser directement en V1 |
|  5 | `service_category`                   | claim                |     ✅    | catégorielle                      |
|  6 | `service_code`                       | claim                |     ✅    | catégorielle                      |
|  7 | `service_units`                      | claim                |     ✅    | numérique                         |
|  8 | `service_date`                       | temporelle           |    ⚠️    | transformer                       |
|  9 | `claim_submission_date`              | temporelle           |    ⚠️    | surtout split temporel            |
| 10 | `claim_submission_timestamp`         | temporelle           |    ⚠️    | split + features temporelles      |
| 11 | `claim_amount`                       | financière           |     ✅    | numérique                         |
| 12 | `requested_reimbursement`            | financière           |     ✅    | numérique                         |
| 13 | `coverage_limit`                     | contrat              |     ✅    | numérique                         |
| 14 | `submission_channel`                 | claim                |     ✅    | catégorielle                      |
| 15 | `document_count`                     | claim                |     ✅    | numérique + missing               |
| 16 | `has_invoice`                        | claim                |     ✅    | booléenne                         |
| 17 | `has_prescription`                   | claim                |     ✅    | booléenne/catégorielle + missing  |
| 18 | `customer_age`                       | client               |     ✅    | numérique                         |
| 19 | `customer_tenure_months`             | client               |     ✅    | numérique                         |
| 20 | `coverage_level`                     | contrat              |     ✅    | catégorielle                      |
| 21 | `customer_behavior_segment`          | synthétique/profil   |    ⚠️    | à examiner                        |
| 22 | `policy_tenure_months`               | contrat              |     ✅    | numérique                         |
| 23 | `recent_policy_change`               | contrat              |     ✅    | booléenne                         |
| 24 | `days_since_policy_change`           | contrat              |     ✅    | numérique                         |
| 25 | `provider_type`                      | provider             |     ✅    | catégorielle                      |
| 26 | `provider_region`                    | provider             |     ✅    | catégorielle                      |
| 27 | `provider_tenure_months`             | provider             |     ✅    | numérique                         |
| 28 | `provider_behavior_segment`          | synthétique/profil   |    ⚠️    | à examiner                        |
| 29 | `days_service_to_submission`         | dérivée              |     ✅    | numérique                         |
| 30 | `reimbursement_ratio`                | dérivée              |     ✅    | numérique                         |
| 31 | `customer_claims_7d`                 | historique           |    ✅*    | très importante                   |
| 32 | `customer_claims_30d`                | historique           |    ✅*    | très importante                   |
| 33 | `customer_claims_90d`                | historique           |    ✅*    | très importante                   |
| 34 | `customer_claims_365d`               | historique           |    ✅*    | très importante                   |
| 35 | `customer_amount_30d`                | historique           |    ✅*    | numérique                         |
| 36 | `customer_amount_365d`               | historique           |    ✅*    | numérique                         |
| 37 | `customer_avg_claim_amount_365d`     | historique           |    ✅*    | numérique                         |
| 38 | `days_since_customer_previous_claim` | historique           |    ✅*    | numérique                         |
| 39 | `days_since_same_provider_claim`     | historique           |    ✅*    | numérique                         |
| 40 | `customer_provider_claims_30d`       | historique           |    ✅*    | très intéressante                 |
| 41 | `same_service_claims_30d`            | historique           |    ✅*    | très intéressante                 |
| 42 | `provider_claims_30d`                | historique           |    ✅*    | provider                          |
| 43 | `provider_claims_90d`                | historique           |    ✅*    | provider                          |
| 44 | `provider_avg_claim_amount_90d`      | historique           |    ✅*    | provider                          |
| 45 | `service_typical_amount`             | référence            |    ✅*    | montant attendu                   |
| 46 | `claim_to_service_median_ratio`      | dérivée              |    ✅*    | signal montant                    |
| 47 | `claim_to_customer_avg_ratio`        | dérivée              |    ✅*    | signal client                     |
| 48 | `claim_to_provider_avg_ratio`        | dérivée              |    ✅*    | signal provider                   |
| 49 | `legitimate_anomaly`                 | simulation           |     ❌    | exclure                           |
| 50 | `legitimate_anomaly_type`            | simulation           |     ❌    | exclure                           |
| 51 | `latent_fraud_score`                 | génération fraude    |    🚫    | leakage                           |
| 52 | `synthetic_fraud_probability`        | génération fraude    |    🚫    | leakage                           |
| 53 | `fraud_difficulty`                   | vérité synthétique   |    🚫    | leakage                           |
| 54 | `fraud_mechanism`                    | vérité synthétique   |    🚫    | leakage                           |
| 55 | `is_fraud`                           | **TARGET**           |    🎯    | y                                 |



In [13]:
TARGET = "is_fraud"

IDENTIFIER_COLUMNS = [
    "claim_id",
    "customer_id",
    "policy_id",
    "provider_id",
]

LEAKAGE_COLUMNS = [
    "legitimate_anomaly",
    "legitimate_anomaly_type",
    "latent_fraud_score",
    "synthetic_fraud_probability",
    "fraud_difficulty",
    "fraud_mechanism",
]

SYNTHETIC_PROFILE_COLUMNS = [
    "customer_behavior_segment",
    "provider_behavior_segment",
]

EXCLUDED_FROM_MODEL = (
    IDENTIFIER_COLUMNS
    + LEAKAGE_COLUMNS
    + SYNTHETIC_PROFILE_COLUMNS
    + [TARGET]
)

candidate_features = [
    column
    for column in claims.columns
    if column not in EXCLUDED_FROM_MODEL
]

print(f"Total columns:             {claims.shape[1]}")
print(f"Identifiers:               {len(IDENTIFIER_COLUMNS)}")
print(f"Leakage/simulation:        {len(LEAKAGE_COLUMNS)}")
print(f"Synthetic profiles:        {len(SYNTHETIC_PROFILE_COLUMNS)}")
print(f"Target:                    1")
print(f"Candidate model features:  {len(candidate_features)}")

Total columns:             55
Identifiers:               4
Leakage/simulation:        6
Synthetic profiles:        2
Target:                    1
Candidate model features:  42
